# PhoBERT Fine-tuning cho Vietnamese Comment Classification

**Mục tiêu:** Train 2 model PhoBERT riêng biệt:
- Showbiz: 6 classes (Phẫn nộ, Cà khịa, Đồng cảm, Ủng hộ, Trung lập, is_trash)
- Education: 5 classes (tích cực, tiêu cực, trung lập, ý kiến riêng, is_trash)

**Yêu cầu:** GPU runtime (T4 hoặc P100)

## Workflow
1. Upload CSV verified lên Colab
2. Install dependencies
3. Train model
4. Evaluate
5. Download checkpoint

## 1. Setup

In [ ]:
!pip install -q transformers torch sentencepiece accelerate scikit-learn pandas openpyxl

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Upload Data

Upload file CSV từ `sampling_labeling/data_labels/`:
- `verify_showbiz_5k.csv`
- `verify_education_5k.csv`

Hoặc nếu đã verify xong, upload file verified.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Config - THAY ĐỔI DOMAIN Ở ĐÂY
DOMAIN = "showbiz"  # "showbiz" hoặc "education"
DATA_PATH = f"verify_{DOMAIN}_5k.csv"  # path tới file vừa upload

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
PATIENCE = 3
MAX_LENGTH = 256
SEED = 42

MODEL_NAME = "vinai/phobert-base-v2"
OUTPUT_DIR = f"/content/phobert_checkpoints/{DOMAIN}"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Domain: {DOMAIN}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

## 3. Load & Prepare Data

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

SHOWBIZ_LABELS = ["Phẫn nộ", "Cà khịa", "Đồng cảm", "Ủng hộ", "Trung lập", "is_trash"]
EDUCATION_LABELS = ["tích cực", "tiêu cực", "trung lập", "ý kiến riêng", "is_trash"]

# Load CSV
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
# Prepare labels
if DOMAIN == "showbiz":
    label_col = "llm_emotion"
    valid_labels = SHOWBIZ_LABELS
else:
    label_col = "llm_stance"
    valid_labels = EDUCATION_LABELS

# Use verified_label if filled, otherwise llm label
if "verified_label" in df.columns:
    df["final_label"] = df["verified_label"].where(
        df["verified_label"].notna() & (df["verified_label"] != ""),
        other=df[label_col]
    )
else:
    df["final_label"] = df[label_col]

# Override trash
df.loc[df["is_trash"] == True, "final_label"] = "is_trash"

# Filter valid labels
df = df[df["final_label"].isin(valid_labels)].reset_index(drop=True)

# Label mapping
label2id = {label: idx for idx, label in enumerate(valid_labels)}
id2label = {idx: label for label, idx in label2id.items()}
df["label_id"] = df["final_label"].map(label2id)

# Clean
df = df.dropna(subset=["text", "label_id"]).reset_index(drop=True)
df["label_id"] = df["label_id"].astype(int)

print(f"\nFinal dataset: {len(df)} samples")
print(f"\nLabel distribution:")
print(df["final_label"].value_counts())

In [ ]:
# Stratified split
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label_id"], random_state=SEED
)
val_ratio = 0.15 / (1 - 0.15)
train_df, val_df = train_test_split(
    train_df, test_size=val_ratio, stratify=train_df["label_id"], random_state=SEED
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nTrain label distribution:")
print(train_df["final_label"].value_counts())

## 4. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = CommentDataset(train_df["text"].tolist(), train_df["label_id"].tolist(), tokenizer)
val_dataset = CommentDataset(val_df["text"].tolist(), val_df["label_id"].tolist(), tokenizer)
test_dataset = CommentDataset(test_df["text"].tolist(), test_df["label_id"].tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

## 5. Model

In [ ]:
import torch.nn as nn
from transformers import AutoModel, get_linear_schedule_with_warmup

class PhoBERTClassifier(nn.Module):
    def __init__(self, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

num_labels = len(label2id)
model = PhoBERTClassifier(num_labels=num_labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded on {device}")
print(f"Num labels: {num_labels}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Class weights for imbalanced data
class_weights = compute_class_weight(
    "balanced", classes=np.unique(train_df["label_id"]), y=train_df["label_id"].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class weights: {dict(zip(valid_labels, class_weights.round(3)))}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")

## 6. Training Loop

In [ ]:
from sklearn.metrics import f1_score

def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1, all_preds, all_labels

In [ ]:
# Training
best_f1 = 0
patience_counter = 0
history = []

print(f"{'='*60}")
print(f"Training {DOMAIN} model ({num_labels} classes)")
print(f"{'='*60}\n")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_f1": val_f1,
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, F1: {val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_f1": best_f1,
        }, f"{OUTPUT_DIR}/best_model.pt")
        print(f"  -> Saved best model (F1={best_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val F1: {best_f1:.4f}")

## 7. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import json

# Load best model
checkpoint = torch.load(f"{OUTPUT_DIR}/best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_f1, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

target_names = [id2label[i] for i in range(num_labels)]

print(f"Test F1 (macro): {test_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(test_labels, test_preds)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print(cm_df)

In [ ]:
# Save config + results
config = {
    "domain": DOMAIN,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LEARNING_RATE,
    "patience": PATIENCE,
    "num_labels": num_labels,
    "label2id": label2id,
    "id2label": id2label,
    "seed": SEED,
}
with open(f"{OUTPUT_DIR}/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

results = {
    "test_f1_macro": float(test_f1),
    "test_loss": float(test_loss),
    "best_val_f1": float(best_f1),
    "classification_report": classification_report(
        test_labels, test_preds, target_names=target_names, output_dict=True
    ),
    "confusion_matrix": cm.tolist(),
    "training_history": history,
}
with open(f"{OUTPUT_DIR}/test_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved config.json and test_results.json to {OUTPUT_DIR}")

## 8. Visualize Training

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = [h["epoch"] for h in history]

# Loss
axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

# F1
axes[1].plot(epochs_range, [h["val_f1"] for h in history], label="Val F1", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Macro")
axes[1].set_title("Validation F1")
axes[1].legend()

# Confusion matrix heatmap
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=axes[2])
axes[2].set_title("Confusion Matrix")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_plots.png", dpi=150)
plt.show()

## 9. Download Checkpoint

In [ ]:
# Zip and download
import shutil
shutil.make_archive(f"/content/phobert_{DOMAIN}", "zip", OUTPUT_DIR)
files.download(f"/content/phobert_{DOMAIN}.zip")
print(f"Download: phobert_{DOMAIN}.zip")

## 10. Batch Inference (Optional)

Sau khi train xong, dùng model để classify toàn bộ 30k comments.

In [ ]:
# Upload all comments CSV (from aggregate_comments_v2.py output)
# uploaded_all = files.upload()

# Run batch inference
# !python phobert-batch-inference.py \
#     --domain {DOMAIN} \
#     --model_dir {OUTPUT_DIR} \
#     --input_csv all_{DOMAIN}_comments.csv \
#     --output_path predictions_{DOMAIN}.csv